In [10]:
import os 
from dotenv import load_dotenv
import pandas as pd 
from phoenix.otel import register 
import phoenix as px
import requests
#create groq llm wrapper
import requests 
from opentelemetry import trace



load_dotenv()


True

In [18]:

class ObservabilityManager:
    def __init__(self, project_name="phoenix-rag-groq"):
        self.session = px.launch_app()
        self.tracer_provider = register(project_name=project_name)
        self.tracer = trace.get_tracer(__name__)
        print("Phoenix UI:", self.session.url)

    def get_tracer(self):
        return self.tracer

In [40]:
import requests

class GroqLLM:
    def __init__(self, tracer):
        self.api_key = os.environ["GROQ_API_KEY"]
        self.tracer = tracer
        self.url = "https://api.groq.com/openai/v1/chat/completions"
        self.model = "llama3-70b-8192"

    def generate(self, prompt):
        with self.tracer.start_as_current_span("groq_llm_call") as span:
            
            headers = {
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json"
            }

            data = {
                "model": self.model,
                "messages": [{"role": "user", "content": prompt}]
            }

            response = requests.post(self.url, headers=headers, json=data)
            
            try:
                result = response.json()
            except Exception:
                raise Exception(f"Invalid JSON response: {response.text}")

            # 🔥 CRITICAL FIX
            if "choices" not in result:
                raise Exception(f"Groq API Error: {result}")

            output = result["choices"][0]["message"]["content"]

            span.set_attribute("input", prompt)
            span.set_attribute("output", output)

            return output

In [41]:
class Retriever:
    def __init__(self, documents):
        self.documents = documents

    def retrieve(self, query, k=2):
        query_words = set(query.lower().split())
        
        scored = []
        for doc in self.documents:
            score = len(query_words & set(doc.lower().split()))
            scored.append((doc, score))
        
        scored.sort(key=lambda x: x[1], reverse=True)
        
        return [doc for doc, _ in scored[:k]]

In [42]:
class RAGPipeline:
    def __init__(self, retriever, llm, tracer):
        self.retriever = retriever
        self.llm = llm
        self.tracer = tracer

    def run(self, query):
        with self.tracer.start_as_current_span("rag_pipeline") as span:
            
            docs = self.retriever.retrieve(query)
            context = "\n".join(docs)

            prompt = f"""
            Answer based only on context:
            Context: {context}
            Question: {query}
            """

            answer = self.llm.generate(prompt)

            span.set_attribute("query", query)
            span.set_attribute("retrieved_docs", context)
            span.set_attribute("answer", answer)

            return {
                "question": query,
                "context": context,
                "answer": answer
            }

In [43]:
class Evaluator:
    def __init__(self, llm):
        self.llm = llm

    def retrieval_score(self, question, context):
        q_words = set(question.lower().split())
        c_words = set(context.lower().split())
        overlap = len(q_words & c_words)
        return 1 if overlap > 1 else 0

    def answer_score(self, question, context, answer):
        prompt = f"""
        Question: {question}
        Context: {context}
        Answer: {answer}

        Is the answer correct based on context?
        Reply only: correct or incorrect
        """
        return self.llm.generate(prompt)

    def evaluate(self, data):
        results = []
        for item in data:
            r_score = self.retrieval_score(item["question"], item["context"])
            a_score = self.answer_score(item["question"], item["context"], item["answer"])

            item["retrieval_score"] = r_score
            item["answer_eval"] = a_score
            results.append(item)

        return results

In [44]:
# Initialize Observability
obs = ObservabilityManager()
tracer = obs.get_tracer()

# Documents
documents = [
    "RAG stands for Retrieval Augmented Generation.",
    "Phoenix is used for LLM observability.",
    "Groq provides ultra-fast LLM inference.",
    "Embeddings convert text into vectors."
]

# Initialize components
retriever = Retriever(documents)
llm = GroqLLM(tracer)
rag = RAGPipeline(retriever, llm, tracer)
evaluator = Evaluator(llm)

# Queries
queries = [
    "What is RAG?",
    "What is Phoenix?",
    "What is Groq?"
]

# Run pipeline
results = []
for q in queries:
    result = rag.run(q)
    results.append(result)
    print(f"\nQ: {q}\nA: {result['answer']}")

Existing running Phoenix instance detected! Shutting it down and starting a new instance...
Overriding of current TracerProvider is not allowed


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: phoenix-rag-groq
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Phoenix UI: http://localhost:6006/


Exception: Groq API Error: {'error': {'message': 'The model `llama3-70b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}